In [ ]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql.functions import (col, sum, avg, count, max, min, rank, dense_rank, row_number, lead, lag, broadcast, explode, flatten, array, struct, lit, udf, collect_list, collect_set)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StringType, IntegerType,
    ArrayType, StructType, StructField
)
from pyspark import StorageLevel

# Use the Python executable from the active environment
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Prefer local loopback for Spark networking
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (
    SparkSession.builder
    .appName("PySparkDemo")
    .master("local[*]")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
print("SparkSession created successfully.")

In [ ]:
customers_data = [
    (1, "Amit"),
    (2, "Neha"),
    (3, "Rahul"),
    (4, "Priya")
]

orders_data = [
    (101, 1, 500),
    (102, 2, 800),
    (103, 1, 300),
    (104, 5, 700)
]

customers_df = spark.createDataFrame(
    customers_data,
    ["CustomerID", "Name"]
)

orders_df = spark.createDataFrame(
    orders_data,
    ["OrderID", "CustomerID", "Amount"]
)

In [ ]:
# Joins

inner_df = customers_df.join(
    orders_df,
    on="CustomerID",
    how="inner"
)

inner_df.show()

In [ ]:
left_df = customers_df.join(
    orders_df,
    on="CustomerID",
    how="left"
)

left_df.show()

In [ ]:
right_df = customers_df.join(
    orders_df,
    on="CustomerID",
    how="right"
)

right_df.show()

In [ ]:
full_df = customers_df.join(
    orders_df,
    on="CustomerID",
    how="full"
)

full_df.show()

In [ ]:
cross_df = customers_df.crossJoin(orders_df)

print("Total rows:", cross_df.count())

In [ ]:
semi_df = customers_df.join(
    orders_df,
    on="CustomerID",
    how="left_semi"
)

semi_df.show()

In [ ]:
anti_df = customers_df.join(
    orders_df,
    on="CustomerID",
    how="left_anti"
)

anti_df.show()

In [ ]:
products_data = [
    (101, "Laptop"),
    (102, "Mouse"),
    (103, "Keyboard")
]

sales_data = [
    (1, 101, 50000),
    (2, 102, 500),
    (3, 103, 1500),
    (4, 101, 52000)
]

products_df = spark.createDataFrame(
    products_data,
    ["ProductID", "ProductName"]
)

sales_df = spark.createDataFrame(
    sales_data,
    ["SaleID", "ProductID", "Amount"]
)

broadcast_df = sales_df.join(
    broadcast(products_df),
    on="ProductID",
    how="inner"
)

broadcast_df.show()

In [ ]:
broadcast_df.explain()

In [ ]:
# Aggregations
sales_data = [
    ("North", "Laptop", 50000),
    ("North", "Mouse", 1000),
    ("South", "Laptop", 60000),
    ("South", "Mouse", 1500),
    ("East", "Laptop", 55000),
    ("East", "Mouse", 1200)
]

sales_df = spark.createDataFrame(
    sales_data,
    ["Region", "Product", "Sales"]
)

In [ ]:
# from pyspark.sql.functions import sum, avg, count

result_df = sales_df.groupBy("Region").agg(
    sum("Sales").alias("TotalSales"),
    avg("Sales").alias("AverageSales"),
    count("*").alias("TransactionCount")
)

result_df.show()

In [ ]:
sales_df.groupBy(
    "Region", "Product"
).agg(
    sum("Sales").alias("TotalSales")
).show()

In [ ]:
rollup_df = sales_df.rollup(
    "Region", "Product"
).agg(
    sum("Sales").alias("TotalSales")
)

rollup_df.show()

In [ ]:
cube_df = sales_df.cube(
    "Region", "Product"
).agg(
    sum("Sales").alias("TotalSales")
)

cube_df.show()

In [ ]:
pivot_data = [
    ("North", "Laptop", 50000),
    ("North", "Mouse", 1000),
    ("South", "Laptop", 60000),
    ("South", "Mouse", 1500)
]

pivot_df = spark.createDataFrame(
    pivot_data,
    ["Region", "Product", "Sales"]
)

In [ ]:
result = pivot_df.groupBy("Region") \
    .pivot("Product", ["Laptop", "Mouse"]) \
    .sum("Sales")

result.show()

In [ ]:
unpivot_df = result.unpivot(
    ids=["Region"],
    values=["Laptop", "Mouse"],
    variableColumnName="Product",
    valueColumnName="Sales"
)

unpivot_df.show()

In [ ]:
# Window Functions
employee_data = [
    (1, "Amit", "IT", 70000),
    (2, "Neha", "IT", 80000),
    (3, "Rahul", "IT", 80000),
    (4, "Priya", "HR", 60000),
    (5, "Kiran", "HR", 65000)
]

employees_df = spark.createDataFrame(
    employee_data,
    ["ID", "Name", "Department", "Salary"]
)

In [ ]:
window_spec = Window.partitionBy(
    "Department"
).orderBy(
    col("Salary").desc()
)

In [ ]:
ranked_df = employees_df \
    .withColumn(
        "Rank",
        rank().over(window_spec)
    ) \
    .withColumn(
        "DenseRank",
        dense_rank().over(window_spec)
    ) \
    .withColumn(
        "RowNumber",
        row_number().over(window_spec)
    )

ranked_df.show()

In [ ]:
sales_data = [
    ("2026-01", 10000),
    ("2026-02", 12000),
    ("2026-03", 15000),
    ("2026-04", 13000)
]

monthly_df = spark.createDataFrame(
    sales_data,
    ["Month", "Sales"]
)

window_spec = Window.orderBy("Month")

comparison_df = monthly_df \
    .withColumn(
        "PreviousSales",
        lag("Sales", 1).over(window_spec)
    ) \
    .withColumn(
        "NextSales",
        lead("Sales", 1).over(window_spec)
    )

comparison_df.show()

In [ ]:
growth_df = comparison_df.withColumn(
    "SalesGrowth",
    col("Sales") - col("PreviousSales")
)

growth_df.show()

In [ ]:
running_window = Window.orderBy(
    "Month"
).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

running_df = monthly_df.withColumn(
    "RunningTotal",
    sum("Sales").over(running_window)
)

running_df.show()

In [ ]:
moving_window = Window.orderBy(
    "Month"
).rowsBetween(-2, 0)

moving_avg_df = monthly_df.withColumn(
    "MovingAverage",
    avg("Sales").over(moving_window)
)

moving_avg_df.show()

In [ ]:
# Advanced Trasformations

data = [
    (1, ["Python", "SQL", "Spark"]),
    (2, ["Java", "Python"]),
    (3, ["SQL", "Azure"])
]

skills_df = spark.createDataFrame(
    data,
    ["EmployeeID", "Skills"]
)

skills_df.show(truncate=False)

In [ ]:
from pyspark.sql.functions import (
    size, array_contains,
    array_distinct, array_sort, array_union
)

result_df = skills_df \
    .withColumn(
        "SkillCount",
        size("Skills")
    ) \
    .withColumn(
        "HasPython",
        array_contains("Skills", "Python")
    ) \
    .withColumn(
        "SortedSkills",
        array_sort("Skills")
    )

result_df.show(truncate=False)

In [ ]:
union_df = skills_df.withColumn(
    "AllSkills",
    array_union(
        col("Skills"),
        array(lit("Python"), lit("AWS"))
    )
)

union_df.show(truncate=False)

In [ ]:
from pyspark.sql.functions import array_intersect

intersect_df = skills_df.withColumn(
    "MatchingSkills",
    array_intersect(
        col("Skills"),
        array(lit("Python"), lit("SQL"))
    )
)

intersect_df.show(truncate=False)

In [ ]:
from pyspark.sql.functions import length, filter as spark_filter

filtered_df = skills_df.withColumn(
    "FilteredSkills",
    spark_filter(
        col("Skills"),
        lambda x: length(x) > 4
    )
)

filtered_df.show(truncate=False)

In [ ]:
from pyspark.sql.functions import transform, upper

transformed_df = skills_df.withColumn(
    "UppercaseSkills",
    transform(
        col("Skills"),
        lambda x: upper(x)
    )
)

transformed_df.show(truncate=False)

In [ ]:
from pyspark.sql.functions import struct

employee_data = [
    (1, "Amit", "IT", 70000),
    (2, "Neha", "HR", 65000)
]

df = spark.createDataFrame(
    employee_data,
    ["ID", "Name", "Department", "Salary"]
)

struct_df = df.withColumn(
    "EmployeeDetails",
    struct(
        col("Name"),
        col("Department"),
        col("Salary")
    )
)

struct_df.select(
    "ID", "EmployeeDetails"
).show(truncate=False)

In [ ]:
exploded_df = skills_df.select(
    "EmployeeID",
    explode("Skills").alias("Skill")
)

exploded_df.show()

In [ ]:
nested_data = [
    (1, [["Python", "SQL"], ["Spark", "Azure"]]),
    (2, [["Java"], ["Docker", "Kubernetes"]])
]

nested_df = spark.createDataFrame(
    nested_data,
    ["ID", "NestedSkills"]
)

flattened_df = nested_df.withColumn(
    "AllSkills",
    flatten("NestedSkills")
)

flattened_df.show(truncate=False)

In [ ]:
df.show()
df.printSchema()

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def salary_category(salary):
    if salary is None:
        return "Unknown"
    elif salary >= 80000:
        return "High"
    elif salary >= 50000:
        return "Medium"
    else:
        return "Low"

salary_udf = udf(
    salary_category,
    StringType()
)

result_df = df.withColumn(
    "SalaryCategory",
    salary_udf(col("Salary"))
)

result_df.show()

In [ ]:
from pyspark.sql.functions import when

result_df = df.withColumn(
    "SalaryCategory",
    when(
        col("Salary").isNull(), "Unknown"
    ).when(
        col("Salary") >= 80000, "High"
    ).when(
        col("Salary") >= 50000, "Medium"
    ).otherwise("Low")
)

result_df.show()

In [ ]:
sales_df.show()

In [ ]:
sales_df.cache()

sales_df.count()  # Materializes the cache

sales_df.groupBy("Region").sum("Sales").show()
sales_df.filter(col("Sales") > 10000).show()

In [ ]:
sales_df.unpersist()

In [ ]:
from pyspark import StorageLevel

sales_df.persist(
    StorageLevel.MEMORY_ONLY
)

sales_df.count()

sales_df.unpersist()

In [ ]:
sales_df.show()

In [ ]:
print(
    sales_df.rdd.getNumPartitions()
)

repartitioned_df = sales_df.repartition(4)

print(
    repartitioned_df.rdd.getNumPartitions()
)

In [ ]:
coalesced_df = repartitioned_df.coalesce(2)

print(
    coalesced_df.rdd.getNumPartitions()
)

In [ ]:
sales_data = [
    (1, "North", 2025, 50000),
    (2, "South", 2025, 60000),
    (3, "North", 2026, 70000),
    (4, "South", 2026, 80000)
]

sales_df = spark.createDataFrame(
    sales_data,
    ["SaleID", "Region", "Year", "Amount"]
)

sales_df.write \
    .mode("overwrite") \
    .partitionBy("Year", "Region") \
    .parquet("output/partitioned_sales")

In [ ]:
df = spark.read.parquet(
    "output/partitioned_sales"
)

df.filter(col("Year") == 2026).show()

In [ ]:
sales_df.write \
    .mode("overwrite") \
    .bucketBy(4, "Region") \
    .sortBy("Region") \
    .saveAsTable("bucketed_sales")

In [ ]:
bucketed_df = spark.table(
    "bucketed_sales"
)

bucketed_df.show()